In [1]:
import pandas as pd

# define important cols to speed up the read process
keep_cols = ['buyer_state', 'drug_name', 'transaction_date', 'calc_base_wt_in_gm']

# chunking
chunks = pd.read_csv(
    "../../data/DEA_2.csv.gz", 
    usecols=keep_cols, 
    chunksize=500000, 
    compression='gzip',
    low_memory=False
)

filtered_results = []

print("Starting extraction...")
for i, chunk in enumerate(chunks):
    mask = chunk['drug_name'].str.upper().isin(['OXYCODONE', 'HYDROCODONE'])
    small_chunk = chunk[mask]
    
    filtered_results.append(small_chunk)
    
    if i % 10 == 0:
        print(f"Processed {i * 500000} rows...")

# save
if filtered_results:
    df_mini = pd.concat(filtered_results)
    df_mini.to_csv('../../data/dea_summary_mini.csv', index=False)
    print(f"Done! Saved {len(df_mini)} rows to 'dea_summary_mini.csv'.")
else:
    print("No data found matching those drug names.")

Starting extraction...
Processed 0 rows...
Processed 5000000 rows...
Processed 10000000 rows...
Processed 15000000 rows...
Processed 20000000 rows...
Processed 25000000 rows...
Processed 30000000 rows...
Processed 35000000 rows...
Processed 40000000 rows...
Processed 45000000 rows...
Processed 50000000 rows...
Processed 55000000 rows...
Processed 60000000 rows...
Processed 65000000 rows...
Processed 70000000 rows...
Processed 75000000 rows...
Processed 80000000 rows...
Processed 85000000 rows...
Processed 90000000 rows...
Processed 95000000 rows...
Processed 100000000 rows...
Processed 105000000 rows...
Processed 110000000 rows...
Processed 115000000 rows...
Processed 120000000 rows...
Processed 125000000 rows...
Processed 130000000 rows...
Processed 135000000 rows...
Processed 140000000 rows...
Processed 145000000 rows...
Processed 150000000 rows...
Processed 155000000 rows...
Processed 160000000 rows...
Processed 165000000 rows...
Processed 170000000 rows...
Processed 175000000 rows.

In [2]:
# inspect the mini file
df = pd.read_csv('../../data/dea_summary_mini.csv')

print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 360395927 entries, 0 to 360395926
Data columns (total 4 columns):
 #   Column              Dtype  
---  ------              -----  
 0   transaction_date    object 
 1   calc_base_wt_in_gm  float64
 2   buyer_state         object 
 3   drug_name           object 
dtypes: float64(1), object(3)
memory usage: 10.7+ GB
None


,transaction_date,calc_base_wt_in_gm,buyer_state,drug_name
0,01/01/2006,0.572708,NJ,HYDROCODONE
1,01/01/2006,1.432305,FL,HYDROCODONE
2,01/01/2006,1.432305,TX,HYDROCODONE
3,01/01/2006,1.432305,TX,HYDROCODONE
4,01/01/2006,2.864610,TX,HYDROCODONE


In [3]:
# Convert to format that can merge with NCHS

# get transaction year
print("Extracting years...")
df['year'] = pd.to_datetime(df['transaction_date']).dt.year

# grouping
print("Summing grams by state and year...")
df_summed = df.groupby(['buyer_state', 'year', 'drug_name'])['calc_base_wt_in_gm'].sum().reset_index()

# pivot so drugs are their own columns
print("Pivoting drug columns...")
df_pivot = df_summed.pivot_table(
    index=['buyer_state', 'year'], 
    columns='drug_name', 
    values='calc_base_wt_in_gm'
).reset_index()

# clean col names
df_pivot.columns.name = None  # Remove the drug_name label from columns
df_pivot = df_pivot.rename(columns={
    'buyer_state': 'state',
    'HYDROCODONE': 'hydro_gms',
    'OXYCODONE': 'oxy_gms'
})

# fill NAs with 0
df_pivot = df_pivot.fillna(0)

# Save final version
df_pivot.to_csv('../../data/dea_final.csv', index=False)
print("Transformation complete! Your file 'arcos_final_summary_06_14.csv' is ready for modeling.")
print(df_pivot.head())

Extracting years...
Summing grams by state and year...
Pivoting drug columns...
Transformation complete! Your file 'arcos_final_summary_06_14.csv' is ready for modeling.
  state  year     hydro_gms        oxy_gms
0    AK  2006  73936.970520  132344.307041
1    AK  2007  80968.424733  141050.065458
2    AK  2008  86069.808517  153192.816393
3    AK  2009  89413.086368  168255.904702
4    AK  2010  90779.457730  174587.019594


In [25]:
import numpy as np
# Data from 2000, 2005
# state mapping dict
state_map = {
    'ALABAMA': 'AL', 'ALASKA': 'AK', 'ARIZONA': 'AZ', 'ARKANSAS': 'AR', 'CALIFORNIA': 'CA',
    'COLORADO': 'CO', 'CONNECTICUT': 'CT', 'DELAWARE': 'DE', 'FLORIDA': 'FL', 'GEORGIA': 'GA',
    'HAWAII': 'HI', 'IDAHO': 'ID', 'ILLINOIS': 'IL', 'INDIANA': 'IN', 'IOWA': 'IA',
    'KANSAS': 'KS', 'KENTUCKY': 'KY', 'LOUISIANA': 'LA', 'MAINE': 'ME', 'MARYLAND': 'MD',
    'MASSACHUSETTS': 'MA', 'MICHIGAN': 'MI', 'MINNESOTA': 'MN', 'MISSISSIPPI': 'MS', 'MISSOURI': 'MO',
    'MONTANA': 'MT', 'NEBRASKA': 'NE', 'NEVADA': 'NV', 'NEW HAMPSHIRE': 'NH', 'NEW JERSEY': 'NJ',
    'NEW MEXICO': 'NM', 'NEW YORK': 'NY', 'NORTH CAROLINA': 'NC', 'NORTH DAKOTA': 'ND', 'OHIO': 'OH',
    'OKLAHOMA': 'OK', 'OREGON': 'OR', 'PENNSYLVANIA': 'PA', 'RHODE ISLAND': 'RI', 'SOUTH CAROLINA': 'SC',
    'SOUTH DAKOTA': 'SD', 'TENNESSEE': 'TN', 'TEXAS': 'TX', 'UTAH': 'UT', 'VERMONT': 'VT',
    'VIRGINIA': 'VA', 'WASHINGTON': 'WA', 'WEST VIRGINIA': 'WV', 'WISCONSIN': 'WI', 'WYOMING': 'WY',
    'DISTRICT OF COLUMBIA': 'DC'
}

# load and clean 2000, 2005 data
df_manual = pd.read_excel("../../data/DEA_2000_2005.xlsx")
df_manual = df_manual.rename(columns={"Year":'year', "State":'state'})
df_manual['state'] = df_manual['state'].str.upper().str.strip().map(state_map)
df_manual = df_manual[['state', 'year', 'hydro_gms', 'oxy_gms']]

df_manual.head()

,state,year,hydro_gms,oxy_gms
0,AK,2000,27018.40,74395.72
1,WV,2000,152541.08,197056.02
2,DE,2000,22179.56,79671.76
3,FL,2000,983039.12,1569180.46
4,CT,2000,99110.22,338763.08


In [26]:
df_manual.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   state      100 non-null    object 
 1   year       100 non-null    int64  
 2   hydro_gms  100 non-null    float64
 3   oxy_gms    100 non-null    float64
dtypes: float64(2), int64(1), object(1)
memory usage: 3.2+ KB


In [36]:
df_2006_16 = pd.read_csv('../../data/dea_final.csv')
df_2006_16 = df_2006_16.loc[df_2006_16.state!="DC"].reset_index(drop=True)
df_2006_16.head()

,state,year,hydro_gms,oxy_gms
0,AK,2006,73936.970520,132344.307041
1,AK,2007,80968.424733,141050.065458
2,AK,2008,86069.808517,153192.816393
3,AK,2009,89413.086368,168255.904702
4,AK,2010,90779.457730,174587.019594


In [37]:
df_manual.head()

,state,year,hydro_gms,oxy_gms
0,AK,2000,27018.40,74395.72
1,WV,2000,152541.08,197056.02
2,DE,2000,22179.56,79671.76
3,FL,2000,983039.12,1569180.46
4,CT,2000,99110.22,338763.08


In [39]:
dea_master = pd.concat([df_manual, df_2006_16], ignore_index=True)
dea_master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 650 entries, 0 to 649
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   state      650 non-null    object 
 1   year       650 non-null    int64  
 2   hydro_gms  650 non-null    float64
 3   oxy_gms    650 non-null    float64
dtypes: float64(2), int64(1), object(1)
memory usage: 20.4+ KB


In [41]:
# 1. Build the grid (901 rows: 53 units * 17 years)
states = dea_master['state'].unique()
years = range(2000, 2017)
full_grid = pd.MultiIndex.from_product([states, years], names=['state', 'year']).to_frame(index=False)

# 2. Merge your data onto the grid
# This creates the 2001-2004 rows but leaves the grams as NaN
dea_full = pd.merge(full_grid, dea_master, on=['state', 'year'], how='left')

# 3. Sort and Fill
dea_full = dea_full.sort_values(['state', 'year'])
dea_full[['hydro_gms', 'oxy_gms']] = dea_full.groupby('state')[['hydro_gms', 'oxy_gms']].transform(lambda x: x.interpolate(method='linear'))

# 4. Check the result
print(dea_full[dea_full['state'] == 'AL'].head(7))

    state  year     hydro_gms       oxy_gms
340    AL  2000  4.742733e+05  2.990482e+05
341    AL  2001  5.390475e+05  3.269180e+05
342    AL  2002  6.038217e+05  3.547878e+05
343    AL  2003  6.685959e+05  3.826576e+05
344    AL  2004  7.333701e+05  4.105274e+05
345    AL  2005  7.981443e+05  4.383972e+05
346    AL  2006  6.506259e+07  1.538785e+06


In [42]:
# Export the final file
dea_full.to_csv('../../data/dea_full_interpolated.csv', index=False)